In [ ]:
# Including system prompt to steer LLAMA-4 output
# RTS, 05 May 2025

# https://www.llama.com/docs/model-cards-and-prompt-formats/llama4/
# https://llama.developer.meta.com/docs/overview/
# https://www.llama.com/docs/model-cards-and-prompt-formats/llama4_omni/
# https://github.com/meta-llama/llama-cookbook/blob/main/getting-started/build_with_llama_4.ipynb
# https://github.com/meta-llama/llama-cookbook/blob/main/getting-started/build_with_llama_api.ipynb
# https://llama.developer.meta.com/docs/rate-limits

In [ ]:
import os
import requests, base64
import time

In [ ]:
#---------  A decorator to calculate the inference time-----------------
def time_it(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} executed in {time.time() - start:.2f}s")
        return result
    return wrapper

In [ ]:
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get('nvidia_key')

In [ ]:
# variables specific to your CoLabsetup ----------------------------------------
from google.colab import drive
import os, sys
drive.mount('/content/drive')
root = '/content/drive/MyDrive/'
sys.path.append(root +"Colab/research/code/")
datapath = root + "Colab/research/data/"
datapathnudge = root + "Projects/Nudge/geo_images/samples/"

Mounted at /content/drive


In [ ]:
"A function takes an image and prompt as inputs and genrates corresponding captions"

@time_it
def generateCaptions_llama4(image_file_path, prompt):
    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
    stream = False # Put stream to False, dont want the model to be a conversational agent for now
    #--------Opens image, base64 encodes it, converting the raw bytes into a text string---------------------
    #--------image_b64 contains the base64 string version of your image----------------
    try:
        with open(image_file_path, "rb") as f:
            image_b64 = base64.b64encode(f.read()).decode()
    except Exception as e:
        print(f"Error processing {image_file_path}: {e}")

    #--------Defining API key------------------------------------------
    #--------API key is given followed by the string "Bearer"

    headers = {
    "Authorization": f"Bearer {os.getenv('NVIDIA_API_KEY')}",
     "Accept": "text/event-stream" if stream else "application/json"
    }

    #--------Model initialization with system prompt --------------------------------------
    # Info on sentinel-2 band operations
    # https://pro.arcgis.com/en/pro-app/3.3/help/analysis/raster-functions/band-arithmetic-function.htm
    # https://www.sciencedirect.com/science/article/pii/S0303243421000507

    SYSTEM_PROMPT = """You are an expert-level assisant for analyzing geospatial satellite imagery from sentinel-2 sources.
    You are keen on identifying possible signs of environmental degradation in satellite imagery. You understand how to interpret
    satellite image bands such as Red, Green, Blue, NearInfrared, and band operations such as Normalized Difference Vegetation Index,
    Normalized Difference Water Index, Burn Area Index, Clay Minerals Index and Ferrous Minerals Index. Describe only the
    environmental conditions observable in the images supplied to you. Do not add generic observatons about landscapes."""

    # TODO:
    # prevent bullent point responses.
    # limit output to 200 words AND complete sentences
    # request good gramar and fine-tuned language use (HOW?)


    payload = {
    "model": 'meta/llama-4-maverick-17b-128e-instruct',
    "messages":
        [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f""" prompt <img src="data:image/png;base64,{image_b64}" />"""}
        ],
    "max_tokens": 512,
    "temperature": 0.7,
    "top_p": 1.00,
    "stream": stream
    }

    response = requests.post(invoke_url, headers=headers, json=payload)
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
dir = datapathnudge
image_files = os.listdir(dir)
print(image_files)

['Paris_2024-11-28.png', 'Islamabad_2025_01_24.png', 'Canberra_openEO_2025-01-13Z_35_149_1738976955.png', 'LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png', 'Damascus_openEO_2025-01-14Z_33_36_1738966392.png']


In [ ]:
prompts = ["What evidence of harmful environmental conditions can be detected in this sentinel-2 satellite image?",
           "What kind of pollution can be detected in this sentinel-2 satellite image?",
           "Is the water safe to drink and the air safe to breath in the area depicted this sentinel-2 satellite image?",
           "Which environmental hazards can be detected in this sentinel-2 satellite image?"]

#prompts = ["What kind of pollution can be detected in this sentinel-2 satellite image?",
#           "Which environmental hazards can be detected in this sentinel-2 satellite image?"]

In [ ]:
 #-----------Main code that generates captions for all the images in the folder across the collection of prompts---------------
for image_file in image_files:
    image_file_path = os.path.join(dir, image_file)
    for prompt in prompts:
        response = generateCaptions_llama4(image_file_path, prompt)
        print(f"{image_file} \n")
        print(response)
        print("--------------------------------------------------------------------------"*2)

generateCaptions_llama4 executed in 4.91s
Paris_2024-11-28.png 

The image shows a densely urbanized area with extensive road networks and limited green spaces. The high concentration of buildings and infrastructure suggests significant environmental impacts, including heat island effects and potential air pollution from vehicular traffic. The presence of a river running through the city indicates a potential source of water, but its condition and any potential pollution are not discernible from the image. Overall, the image highlights the environmental challenges associated with dense urbanization.
----------------------------------------------------------------------------------------------------------------------------------------------------
generateCaptions_llama4 executed in 19.96s
Paris_2024-11-28.png 

The image shows a densely urbanized area with extensive road networks and limited green spaces. The high concentration of buildings and infrastructure suggests significant enviro